In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator
from matplotlib.patches import Rectangle
from matplotlib.colors import Normalize
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

In [11]:
def grid(N):
  """
  Creates grid of points N x N divisions.
  """
  # Create grid
  x = np.linspace(-1.0, 1.0, N)
  y = np.linspace(-1.0, 1.0, N)
  X, Y = np.meshgrid(x, y)
  dx = x[1] - x[0]

  return x, y, X, Y, dx

In [12]:
def idx(i, j):
  "Flattens grid into an array by converting 2D coordinates into a 1D index."
  return i * N + j

In [ ]:
def assign_points(N, dx, plate_sep, plate_half_len):
  """
  Assigns values to points on a grid. Defines boundary conditions then assigns
  them to correct points. All other points are assigned a value using the
  finite difference form:

  V[i,j] = 0.25 * (V[i-1, j] + V[i+1, j] + V[i, j-1] + V[i, j+1])
  
  which just states that the value of a point is the average of its four
  neighbors, making sure grad^2(V) = Laplaces equation holds.
  """

  # Convert coordinates into grid indices
  ix_pos = int(round((plate_sep + 1.0) / dx))
  ix_neg = int(round((-plate_sep + 1.0) / dx))
  iy_lo = int(round((-plate_half_len + 1.0) / dx))
  iy_hi = int(round(( plate_half_len + 1.0) / dx))

  ix_pos = np.clip(ix_pos, 1, N - 2)
  ix_neg = np.clip(ix_neg, 1, N - 2)
  iy_lo = np.clip(iy_lo, 0, N - 1)
  iy_hi = np.clip(iy_hi, 0, N - 1)

  # Boundary conditions
  BC = {}

  # Outer box
  for j in range(N):
    BC[idx(0, j)] = 0.0
    BC[idx(N - 1, j)] = 0.0
  for i in range(N):
    BC[idx(i, 0)] = 0.0
    BC[idx(i, N - 1)] = 0.0

  # Plates
  for i in range(iy_lo, iy_hi + 1):
    BC[idx(i, ix_pos)] = 1.0
    BC[idx(i, ix_neg)] = -1.0

  # Degrees of freedom; total number of unknown points
  n_dof = N * N
  # Matrix N^2 x N^2
  A = lil_matrix((n_dof, n_dof), dtype=float)
  b = np.zeros(n_dof)

  # Assign values
  for i in range(N):
    for j in range(N):
      k = idx(i, j)

      # Apply boundary conditions
      if k in BC:
        A[k, k] = 1.0
        b[k] = BC[k]

      # Assign values to unknown points, each point's potential must be the
      # average of its four neighbors. This requires grad^2(V) = 0
      else:
        A[k, k] = -4.0 # Center
        A[k, idx(i - 1, j)] =  1.0 # Up
        A[k, idx(i + 1, j)] =  1.0 # Down
        A[k, idx(i, j - 1)] =  1.0 # Left
        A[k, idx(i, j + 1)] =  1.0 # Right

  return A, b, ix_pos, ix_neg, iy_lo, iy_hi

In [5]:
def solve_capacitor(N, dx, plate_half_len=0.4, plate_sep=0.2):

  V_flat = spsolve(A.tocsr(), b)
  V = V_flat.reshape(N, N)

  Ey, Ex = np.gradient(-V, dx, dx)
  E_mag  = np.sqrt(Ex**2 + Ey**2)
  return V, Ex, Ey, E_mag


def draw_plates(ax, x, y, ixp, ixn, iy0, iy1, lw=2.5):
    ax.plot([x[ixp]]*2, [y[iy0], y[iy1]], "k-", lw=lw, solid_capstyle="butt")
    ax.plot([x[ixn]]*2, [y[iy0], y[iy1]], "k-", lw=lw, solid_capstyle="butt")


def safe_norm_vmax(arr, pct=95):
    """Percentile-based vmax ignoring NaN/zero."""
    valid = arr[np.isfinite(arr) & (arr > 0)]
    if len(valid) == 0:
        return 1.0
    return float(np.percentile(valid, pct))


def quiver_subsample(X, Y, Ex, Ey, E_mag, step):
    """Return subsampled, unit-normalised quiver arrays."""
    xq  = X[::step, ::step]
    yq  = Y[::step, ::step]
    Exq = Ex[::step, ::step]
    Eyq = Ey[::step, ::step]
    Eq  = E_mag[::step, ::step]
    with np.errstate(invalid="ignore", divide="ignore"):
        Exh = np.where(Eq > 0, Exq / Eq, 0.0)
        Eyh = np.where(Eq > 0, Eyq / Eq, 0.0)
    return xq, yq, Exh, Eyh, Eq


# ── Solve reference geometry ──────────────────────────────────────────────────
N = 200
x, y, X, Y, V, Ex, Ey, E_mag, ixp, ixn, iy0, iy1 = solve_capacitor(N=N)


# ══════════════════════════════════════════════════════════════════════════════
# Figure 1 – Potential contours (left) + full-domain quiver (right)
# ══════════════════════════════════════════════════════════════════════════════
xq, yq, Exh, Eyh, Eq = quiver_subsample(X, Y, Ex, Ey, E_mag, step=12)

fig1, axes = plt.subplots(1, 2, figsize=(11, 4.8), facecolor="white")
fig1.suptitle("Parallel-Plate Capacitor – Potential and Electric Field Vectors", fontsize=13)

ax = axes[0]
cf = ax.contourf(X, Y, V, levels=40, cmap="RdBu_r")
ax.contour(X, Y, V, levels=20, colors="k", linewidths=0.4, alpha=0.45)
fig1.colorbar(cf, ax=ax, label="Potential V", shrink=0.85)
ax.set_title("Electric Potential")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_aspect("equal")
draw_plates(ax, x, y, ixp, ixn, iy0, iy1)
ax.set_xlim(-1, 1); ax.set_ylim(-1, 1)

ax2 = axes[1]
vmax1 = safe_norm_vmax(Eq)
norm1 = Normalize(vmin=0, vmax=vmax1)
qv = ax2.quiver(xq, yq, Exh, Eyh, Eq,
                cmap="inferno", norm=norm1,
                angles="xy", scale=28, width=0.004,
                headwidth=4, headlength=5, alpha=0.85)
fig1.colorbar(qv, ax=ax2, label="|E|  (arb. units)", shrink=0.85)
ax2.set_title("Electric Field Vectors\n(uniform length, colour = |E|)")
ax2.set_xlabel("x"); ax2.set_ylabel("y")
ax2.set_aspect("equal")
draw_plates(ax2, x, y, ixp, ixn, iy0, iy1)
ax2.set_xlim(-1, 1); ax2.set_ylim(-1, 1)

fig1.tight_layout()



# ══════════════════════════════════════════════════════════════════════════════
# Figure 2 – Zoomed quiver near the positive plate edge (fringing region)
# ══════════════════════════════════════════════════════════════════════════════
x_plate = x[ixp]
xlo, xhi = x_plate - 0.30, x_plate + 0.55
ylo, yhi = -0.65, 0.65

mask_xi = (x >= xlo) & (x <= xhi)
mask_yi = (y >= ylo) & (y <= yhi)
idx = np.ix_(mask_yi, mask_xi)

X_z  = X[idx];  Y_z  = Y[idx]
Ex_z = Ex[idx]; Ey_z = Ey[idx]; Em_z = E_mag[idx]

xqz, yqz, Exhz, Eyhz, Eqz = quiver_subsample(X_z, Y_z, Ex_z, Ey_z, Em_z, step=7)

fig2, ax = plt.subplots(figsize=(7, 5.2), facecolor="white")
vmax2 = safe_norm_vmax(Eqz)
norm2 = Normalize(vmin=0, vmax=vmax2)
qv2 = ax.quiver(xqz, yqz, Exhz, Eyhz, Eqz,
                cmap="inferno", norm=norm2,
                angles="xy", scale=20, width=0.006,
                headwidth=4, headlength=5, alpha=0.9)
fig2.colorbar(qv2, ax=ax, label="|E|  (arb. units)")
ax.plot([x[ixp]]*2, [y[iy0], y[iy1]], "k-", lw=3, solid_capstyle="butt",
        label="Plate (V = +1)", zorder=5)
ax.axvline(x_plate, color="gray", ls="--", lw=0.8, alpha=0.6, label="Plate edge")
ax.set_xlim(xlo, xhi); ax.set_ylim(ylo, yhi)
ax.set_title("Zoomed Fringing Field Near Positive Plate Edge")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_aspect("equal")
ax.legend(fontsize=9, loc="upper right")
fig2.tight_layout()



# ══════════════════════════════════════════════════════════════════════════════
# Figure 3 – |E| along y = 0: interior vs. fringing
# ══════════════════════════════════════════════════════════════════════════════
iy_mid = N // 2
E_along_x = E_mag[iy_mid, :]
mask_between = (x >= x[ixn]) & (x <= x[ixp])
mask_outside = x > x[ixp]

fig3, ax = plt.subplots(figsize=(7, 4), facecolor="white")
ax.plot(x[mask_between], E_along_x[mask_between], color="#1f77b4",
        label="Between plates (interior)")
ax.plot(x[mask_outside], E_along_x[mask_outside], color="#d62728",
        label="Outside (fringing)")
ax.axvline(x[ixp], color="k", ls="--", lw=1, label="Plate edge")
ax.set_xlabel("x  (along midline y = 0)")
ax.set_ylabel("|E|  (arb. units)")
ax.set_title("Electric Field Magnitude Along Horizontal Midline (y = 0)")
ax.legend(fontsize=9)
ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
fig3.tight_layout()


# ══════════════════════════════════════════════════════════════════════════════
# Figure 4 – Fringing & interior field vs. plate separation
# ══════════════════════════════════════════════════════════════════════════════
separations  = np.linspace(0.05, 0.45, 10)
obs_offset   = 0.08
E_fringe_sep = []
E_int_sep    = []

for sep in separations:
    xi, yi, Xi, Yi, Vi, Exi, Eyi, Ei, ixpi, ixni, iy0i, iy1i = solve_capacitor(
        N=100, plate_half_len=0.30, plate_sep=sep)
    iy_mdi = len(yi) // 2
    x_obs  = xi[ixpi] + obs_offset
    if x_obs > xi[-1]:
        E_fringe_sep.append(np.nan)
    else:
        ix_obs = np.argmin(np.abs(xi - x_obs))
        E_fringe_sep.append(Ei[iy_mdi, ix_obs])
    ix_mid = (ixpi + ixni) // 2
    E_int_sep.append(Ei[iy_mdi, ix_mid])

E_fringe_sep = np.array(E_fringe_sep)
E_int_sep    = np.array(E_int_sep)
full_sep     = 2 * separations
E_ideal      = 2.0 / full_sep

fig4, ax = plt.subplots(figsize=(7, 4.5), facecolor="white")
ax.plot(full_sep, E_fringe_sep, "o-", color="#d62728",
        label=f"Fringing (x = plate + {obs_offset})")
ax.plot(full_sep, E_int_sep,   "s-", color="#1f77b4",
        label="Interior (midpoint)")
ax.plot(full_sep, E_ideal,     "k--", label="Ideal: E = V/d")
ax.set_xlabel("Plate separation  d  (arb. units)")
ax.set_ylabel("|E|  (arb. units)")
ax.set_title("Field Magnitude vs. Plate Separation")
ax.legend(fontsize=9)
ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
fig4.tight_layout()


# ══════════════════════════════════════════════════════════════════════════════
# Figure 5 – Interior field profile vs. ideal value
# ══════════════════════════════════════════════════════════════════════════════
ix_mid_r  = (ixp + ixn) // 2
E_int_y   = E_mag[:, ix_mid_r]
E_ideal_v = 2.0 / (2 * 0.15)
y_lo_p    = y[iy0]
y_hi_p    = y[iy1]

fig5, ax = plt.subplots(figsize=(7, 4), facecolor="white")
ax.plot(y, E_int_y, color="#1f77b4", label="Numerical |E(y)| at x = 0")
ax.axhline(E_ideal_v, color="k", ls="--", lw=1.5,
           label=f"Ideal E = V/d = {E_ideal_v:.2f}")
ax.axvline(y_lo_p, color="gray", ls=":", lw=1)
ax.axvline(y_hi_p, color="gray", ls=":", lw=1, label="Plate ends")
ax.fill_betweenx([0, E_ideal_v * 1.6], y_lo_p, y_hi_p,
                 color="lightblue", alpha=0.3, label="Plate extent")
ax.set_xlabel("y  (along midplane x = 0)")
ax.set_ylabel("|E|  (arb. units)")
ax.set_title("Interior Field Profile vs. Ideal Parallel-Plate Value")
ax.set_xlim(-1, 1); ax.set_ylim(bottom=0)
ax.legend(fontsize=9)
ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
fig5.tight_layout()


IndentationError: unindent does not match any outer indentation level (<tokenize>, line 25)